In [3]:
import pandas as pd
import re
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from imblearn.over_sampling import RandomOverSampler

# Step 1: Load and Clean the Dataset
data = pd.read_csv("Resume.csv")
data = data.dropna()

# Check actual column names (for debug)
print("Columns in dataset:", data.columns)

# Function to clean the resume text
def clean_text(text):
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)
    text = re.sub(r'\W', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.lower()

# Apply cleaning to the correct column
data['cleaned_resume'] = data['Resume'].apply(clean_text)

# Step 2: Feature and Label Selection
X = data['cleaned_resume']
y = data['Category']

# Step 3: Vectorization
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_vectorized = vectorizer.fit_transform(X)

# Step 4: Balance the Dataset (if classes are imbalanced)
ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X_vectorized, y)

# Step 5: Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

# Step 6: Train the Model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Step 7: Evaluate the Model
y_pred = model.predict(X_test)
print("\nModel Evaluation Report:\n")
print(classification_report(y_test, y_pred))

# Step 8: Save the Model and Vectorizer
with open("resume_classifier.pkl", "wb") as f:
    pickle.dump(model, f)

with open("vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

print("\n✅ Model training complete and files saved as 'resume_classifier.pkl' and 'vectorizer.pkl'")


Columns in dataset: Index(['Category', 'Resume'], dtype='object')

Model Evaluation Report:

                           precision    recall  f1-score   support

                 Advocate       1.00      1.00      1.00        13
                     Arts       1.00      1.00      1.00        20
       Automation Testing       0.94      1.00      0.97        15
               Blockchain       1.00      1.00      1.00        13
         Business Analyst       1.00      1.00      1.00        19
           Civil Engineer       1.00      1.00      1.00        17
             Data Science       1.00      1.00      1.00        12
                 Database       1.00      1.00      1.00        24
          DevOps Engineer       1.00      0.88      0.94        17
         DotNet Developer       0.95      1.00      0.97        19
            ETL Developer       1.00      1.00      1.00        19
   Electrical Engineering       1.00      1.00      1.00        21
                       HR       1.0